# Embed a search query

Vector search compares two vectors: the **review vectors** (already stored in `review.embedding`, loaded from `theater_reviews_with_vectors.sql`) and the **query vector** for the text being searched. This notebook makes the query vector.

It embeds a search sentence with the same model as the reviews (**sentence-transformers/all-MiniLM-L6-v2**, 384 dimensions, unit length). It then replaces every `:query_vector` placeholder in a SQL file with that vector and saves a ready-to-run copy.

**Steps:** run the cells top to bottom. To search for something else, change `QUERY_TEXT` in step 2 and run the cells again.

The first run downloads the model (about 90 MB). It runs on CPU and needs no paid service.

## 1. Install dependencies (once)

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: C:\Users\dylan\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Choose the search text and SQL file

- `QUERY_TEXT`: what the user is searching for, in plain English.
- `SQL_FILE`: a query that contains `:query_vector`.
- `OUT_FILE`: where to save the filled-in copy.

In [2]:
from pathlib import Path

QUERY_TEXT = "comfortable, accessible venue with a clear view of the stage"

SQL_FILE = Path("../SQL_Scripts/part_d_combined_query.sql")
OUT_FILE = Path("../SQL_Scripts/part_d_combined_query_filled.sql")

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
PLACEHOLDER = ":query_vector"

## 3. Embed the search text

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)

# normalize_embeddings=True gives unit-length vectors, matching the stored
# review vectors, so cosine distance (<=>) ranks them correctly.
vector = model.encode(QUERY_TEXT, normalize_embeddings=True)

print("Dimensions:", len(vector))                       # expect 384
print("Norm:      ", round(float(np.linalg.norm(vector)), 4))  # expect 1.0
assert len(vector) == 384, "Model output does not match vector(384)"

# pgvector literal, quoted so it can replace :query_vector directly.
query_vector = "'[" + ",".join(f"{x:.6f}" for x in vector) + "]'"
print("Literal:   ", query_vector[:80], "...")

C:\Users\dylan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3351.30it/s]


Dimensions: 384
Norm:       1.0
Literal:    '[0.117627,0.003914,-0.017215,-0.034629,-0.025622,0.052036,0.061161,-0.013044,0. ...


## 4. Write the ready-to-run SQL file

This replaces every `:query_vector` in `SQL_FILE` (the main query and the `EXPLAIN`) and saves the result to `OUT_FILE`. Open that file in pgAdmin or VS Code and run it.

In [4]:
sql = SQL_FILE.read_text(encoding="utf-8")
count = sql.count(PLACEHOLDER)
assert count > 0, f"No {PLACEHOLDER} found in {SQL_FILE}"

header = f"-- Generated by embed_query.ipynb with {MODEL_NAME}\n-- Query text: {QUERY_TEXT}\n\n"
OUT_FILE.write_text(header + sql.replace(PLACEHOLDER, query_vector), encoding="utf-8")

print(f"Replaced {count} placeholder(s). Wrote {OUT_FILE.resolve()}")

Replaced 2 placeholder(s). Wrote C:\Users\dylan\repos\adv-db-ui-semester-project\Phase3_OO_JSONB_Vector\SQL_Scripts\part_d_combined_query_filled.sql
